> **Status — forward-looking spec, not runnable today.**
>
> This notebook is a *target* for what a block's `report.ipynb` should look like once the framework is filled out. Most of what it imports (`framework.project`, `framework.render`, `framework.plotting`, `framework.analysis.sweep`) is several implementation steps away.
>
> | Import | Status |
> |---|---|
> | `framework.units` symbols (`V`, `A`, `mA`, `mW`, `degC`, `Ohm`, …) | **Available** (step 1) |
> | `Constant`, `Quantity`, `RangeQuantity` | **Available** (step 1) |
> | `framework.project` (config loaders, DAG runner) | step 2 – step 4 |
> | `framework.render` (header / status / contracts / verifications tables) | step 12 |
> | `framework.plotting.quantity_to_plot` | step 11 – 12 |
> | `framework.analysis.sweep` | step 11 |
>
> For a notebook that actually runs on what step 1 ships, see [`hw_analysis_framework/notebooks/quantity_and_units_demo.ipynb`](hw_analysis_framework/notebooks/quantity_and_units_demo.ipynb).
>
> Treat this file as a design artifact alongside `hardware_analysis_framework_design.md` — it pins down what the engineer-facing surface should feel like before the underlying machinery is written.


# CAN Transceiver Block — Analysis Report

**Template:** This is the standard `report.ipynb` template, populated for a CAN transceiver block (`blocks/can_transceiver/`). Adapt the **Key Analyses** section to your block's specifics; the surrounding sections are auto-generated by the framework and should not be manually edited.

**Execution pattern:** Cell 1 runs the full analysis DAG with caching. Every other cell just *renders* pre-computed results from the `results` object — no further computation. This makes the notebook fast to re-run and immune to stateful-cell ordering bugs.

**Reproducibility:** Run this notebook with a fresh kernel before committing. The framework provides `framework.run_notebook(path, kernel='fresh')` for this; it is also what the design-review artifact generator uses to produce the HTML and PDF.

**Block owner:** *(from `blocks/can_transceiver/README.md` — populated automatically by `render.header`)*

In [ ]:
# Cell 1 — All computation happens here.
# Everything below this cell is pure rendering of `results`.

from framework import project, render
from framework.units import V, A, mA, uA, mW, degC, Ohm

# Load project context: scenarios, modes, requirements, netlist, component library
project.load()

# Execute the analysis DAG across all (scenario, mode) pairs.
# Cached results return immediately when inputs haven't changed.
results = project.run(block="can_transceiver")

## Report Header

Block name, owner, generated timestamp, framework version, component library version, netlist file hash, and git commit hash. Reproducibility metadata — anyone reading this report in five years can rebuild the inputs.

In [ ]:
render.header(block="can_transceiver", results=results)

## Status

Green / yellow / red traffic light. One glance tells you the block's state. Yellow = warnings (non-critical test failures, soft-margin violations). Red = critical failures or contract violations.

In [ ]:
render.status_banner(results)

## Contracts Published by This Block

Every Contract this block exports, with declared values per mode, actual computed values per mode, and the gap. Highlights contracts running close to their declared boundary (where actual is within 10% of the declared edge in any scenario).

In [ ]:
render.contracts_table(results, block="can_transceiver")

# Example output:
#
# Contract           Mode         Declared              Actual                Margin   Status
# can_5v_draw        off          0 mA                  0 mA                  —        ✓
# can_5v_draw        sleep        5–15 µA               6–12 µA               OK       ✓
# can_5v_draw        active       50–75 mA              52–73 mA              OK       ✓
# can_5v_draw        diagnostic   50–75 mA              74 mA in hot_high_vin TIGHT    ⚠
# can_5v_draw        calibration  N/A                   —                     —        —

## Verification Test Results

Every `@verification_test` registered to this block. Each test shows pass/fail, the failing scenario/mode (if any), the margin to spec, the evidence dict, and a link to the Jama requirement it covers.

In [ ]:
render.verification_results(results, block="can_transceiver")

# Example output (rendered as expandable cards in HTML, flattened tables in PDF):
#
# ✓ test_termination_impedance       REQ-CAN-005   CRITICAL
#   Differential termination within 108 Ω – 132 Ω across all scenarios.
#   Margin: +6.7 Ω at cold_low_vin.
#
# ✓ test_junction_temp_max           REQ-THM-014   CRITICAL
#   U501 junction stays below 125 °C across all (scenario × mode).
#   Margin: 23 °C at hot_high_vin / diagnostic.
#
# ⚠ test_sleep_quiescent             REQ-PWR-014   WARNING
#   Sleep current within 15 µA budget — declared margin tight under hot_high_vin (1 µA).
#
# ✓ test_bus_loading_share           REQ-CAN-018   INFO
#   Active-mode bus loading 10–25 % of bus capacity.

## Key Analyses

*Engineer-curated section. Customize the plots and tables below to highlight what matters most for this block. The standard sections above (header, status, contracts, verifications, coverage, consumers, provenance) are auto-generated and should not be edited here.*

### Current draw vs mode

In [ ]:
from framework.plotting import quantity_to_plot

fig = quantity_to_plot.bar_by_mode(
    quantity=results["can_5v_draw"],
    title="CAN Subsystem 5V Current Draw by Mode",
    show_range=True,           # min/max whiskers on each bar
    scenario="hot_high_vin",   # render this scenario; toggleable in HTML
)
fig.show()

### Junction temperature heatmap across (scenario × mode)

In [ ]:
fig = quantity_to_plot.heatmap(
    quantity=results["transceiver_junction_temp"],
    title="U501 Junction Temperature (°C)",
    x_axis="scenario",
    y_axis="mode",
    color_max_warn=110,    # yellow above 110 °C
    color_max_crit=125,    # red above 125 °C (datasheet absolute max)
)
fig.show()

### Custom: dissipation vs ambient temperature, active mode

Sanity check — verify thermal margin doesn't degrade unacceptably at the hot end of the operating envelope. Uses the framework's `sweep` helper to evaluate `transceiver_dissipation` across a continuous ambient range, rather than only at the discrete defined scenarios.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from framework.analysis import sweep

ambients = np.linspace(-40, 85, 50) * degC
dissipations = sweep(
    results["transceiver_dissipation"],
    over={"ambient_temp": ambients},
    fixed={"mode": "active", "scenario": "nominal_supply"},
).to(mW)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ambients.magnitude,
    y=dissipations.magnitude,
    mode="lines",
    name="Dissipation",
))
fig.update_layout(
    title="U501 Static Dissipation vs Ambient Temperature (Active Mode)",
    xaxis_title="Ambient (°C)",
    yaxis_title="P (mW)",
)
fig.show()

### Termination impedance vs scenario

R501 + R502 series split termination. CAN spec requires 108–132 Ω differential at every scenario, including resistor tolerance and temperature coefficient drift.

In [ ]:
fig = quantity_to_plot.bar_by_scenario(
    quantity=results["termination_resistance"],
    title="Differential Termination R501 + R502",
    spec_min=108 * Ohm,
    spec_max=132 * Ohm,
    show_range=True,
)
fig.show()

## Requirement Coverage

Which Jama requirements this block tests. Surfaces coverage gaps explicitly — if a requirement is tagged to this block but no verification covers it, it appears here with a red flag.

In [ ]:
render.requirement_coverage(results, block="can_transceiver")

# Example output:
#
# Requirement      Description                                       Covered by                       Status
# REQ-CAN-005      Differential termination 108–132 Ω                 test_termination_impedance       ✓
# REQ-CAN-018      Bus loading ≤ 30 % of bus capacity                 test_bus_loading_share           ✓
# REQ-PWR-014      Sleep current budget 15 µA                         test_sleep_quiescent             ⚠ tight
# REQ-THM-014      Junction temp ≤ 125 °C                             test_junction_temp_max           ✓
# REQ-CAN-022      ESD tolerance ±15 kV HBM                           — none —                         ✗ uncovered

## Cross-Block Consumers

**Auto-generated by walking the DAG forward** — for each Contract this block publishes, the framework finds every analysis function in other blocks that depends on it, and reports which block they belong to. Useful for impact analysis when this block's contracts change.

In [ ]:
render.cross_block_consumers(block="can_transceiver")

# Example output:
#
# Contract         Consumed by
# can_5v_draw      blocks.power_supply.total_5v_load
#                  blocks.power_supply.regulator_efficiency
#                  project.verifications.system_power_budget
# can_5v_draw      (no consumers — orphan contract; consider removing or documenting why)

## Provenance Drill-Down

Interactive tree for tracing any Quantity back to its root inputs. In HTML, click a node to expand its children. In PDF, the tree is rendered flat with a configurable depth limit (default 4).

In [ ]:
render.provenance_explorer(
    quantity=results["transceiver_junction_temp"],
    scenario="hot_high_vin",
    mode="diagnostic",
    max_depth=6,
)

# Example (rendered as a Plotly tree in HTML, indented text in PDF):
#
# transceiver_junction_temp = 102 °C
# ├── ambient_temp = 85 °C  (from scenario "hot_high_vin")
# └── transceiver_dissipation = 0.34 W
#     ├── transceiver_supply_current = 68 mA  (leaf, mode="diagnostic")
#     └── vsupply_5v = 5.05 V  (Contract from blocks.power_supply)
#         ├── (declared boundary: 4.75–5.25 V — within range ✓)
#         └── ...